In [1]:
"hewlo"


'hewlo'

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

In [13]:
# Start Spark session
spark = SparkSession.builder.appName("sparkSession").getOrCreate()
print("here we go")

here we go


In [14]:
# I'm running Jupyter on my Windows Command Prompt (cuz google dataproc credits and Docker is being stinky)
file_path = r"C:\Users\mpeng\Downloads\2019-04.csv"

df = spark.read.csv(file_path, header=True, inferSchema=True)

selected_df = df.select("passenger_count", "puLocationid", "doLocationid", "total_amount")

selected_df.show(10)

+---------------+------------+------------+------------+
|passenger_count|puLocationid|doLocationid|total_amount|
+---------------+------------+------------+------------+
|            1.0|       239.0|       239.0|         8.8|
|            1.0|       230.0|       100.0|         8.3|
|            1.0|        68.0|       127.0|       47.75|
|            1.0|        68.0|        68.0|         7.3|
|            1.0|        50.0|        42.0|       23.15|
|            1.0|        95.0|       196.0|         9.8|
|            1.0|       211.0|       211.0|         6.8|
|            1.0|       237.0|       162.0|         7.8|
|            1.0|       148.0|        37.0|        20.3|
|            1.0|       265.0|       265.0|        0.31|
+---------------+------------+------------+------------+
only showing top 10 rows


In [15]:
assembler = VectorAssembler(
    inputCols=["passenger_count", "puLocationid", "doLocationid"],
    outputCol="features"
)

dtr = DecisionTreeRegressor(featuresCol="features", labelCol="total_amount")

pipeline = Pipeline(stages=[assembler, dtr])

In [16]:
trainDF, testDF = selected_df.randomSplit([0.7, 0.3], seed=42)
model = pipeline.fit(trainDF)
predictions = model.transform(testDF)
selections = predictions.select("passenger_count", "PULocationID", "DOLocationID", "total_amount", "prediction")
selections.show(10)

+---------------+------------+------------+------------+------------------+
|passenger_count|PULocationID|DOLocationID|total_amount|        prediction|
+---------------+------------+------------+------------+------------------+
|            0.0|         3.0|       136.0|        22.8| 17.78132737051682|
|            0.0|         4.0|        75.0|       28.55| 15.78151960444656|
|            0.0|         4.0|        87.0|       16.56| 15.78151960444656|
|            0.0|         4.0|        87.0|        16.8| 15.78151960444656|
|            0.0|         4.0|       113.0|       12.85| 15.78151960444656|
|            0.0|         4.0|       144.0|        11.3| 17.78132737051682|
|            0.0|         4.0|       148.0|         8.8| 17.78132737051682|
|            0.0|         4.0|       234.0|        11.0| 17.78132737051682|
|            0.0|         7.0|         7.0|         5.3|21.257667479391593|
|            0.0|         7.0|       121.0|        28.8| 17.78132737051682|
+-----------

In [17]:
evaluator = RegressionEvaluator(
    labelCol="total_amount",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator.evaluate(predictions)
print(f"RMSE: {rmse:.2f}")

RMSE: 14.08
